# Proyecto - Asistente de IA para aerolínea

Ahora vamos a reunir lo que hemos aprendido para crear un asistente de IA de atención al cliente para una aerolínea

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Inicialización

load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"La clave de API de Google existe y comienza con {google_api_key[:8]}")
else:
    print("La clave de API de Google no está configurada")

MODEL = "gemini-3.1-flash-lite"
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# Como alternativa, si quieres usar Ollama en lugar de Gemini
# Verifica que Ollama esté corriendo localmente (ver el ejercicio de week1/day2) y luego descomenta estas 2 líneas
# MODEL = "llama3.2"
# gemini = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
system_message = """
Eres un asistente útil para una aerolínea llamada FlightAI.
Da respuestas breves y amables, de no más de 1 oración.
Sé siempre preciso. Si no sabes la respuesta, dilo.
"""

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

## Herramientas

Las herramientas son una función increíblemente poderosa que ofrecen los LLMs de frontera.

Con las herramientas, puedes escribir una función y hacer que el LLM la llame como parte de su respuesta.

¿Suena casi espeluznante... le estamos dando el poder de ejecutar código en nuestra máquina?

Bueno, más o menos.

In [ ]:
# Empecemos creando una función útil

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Herramienta llamada para la ciudad {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Precio de boleto desconocido")
    return f"El precio de un boleto a {destination_city} es {price}"


In [ ]:
get_ticket_price("London")

In [ ]:
# Hay una estructura de diccionario particular que se requiere para describir nuestra función:

price_function = {
    "name": "get_ticket_price",
    "description": "Obtiene el precio de un boleto de ida y vuelta a la ciudad de destino.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "La ciudad a la que el cliente quiere viajar",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# Y esto se incluye en una lista de herramientas:

tools = [{"type": "function", "function": price_function}]

In [ ]:
tools

## Haciendo que Gemini use nuestra herramienta

Hay algunos detalles delicados para permitir que Gemini "llame a nuestra herramienta"

Lo que hacemos en realidad es darle al LLM la oportunidad de informarnos que quiere que ejecutemos la herramienta.

Así se ve la nueva función de chat:

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = gemini.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
# Tenemos que escribir esa función handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Hagamos un par de mejoras

Manejar múltiples llamadas a herramientas en una sola respuesta

Manejar múltiples llamadas a herramientas una tras otra

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
import sqlite3


In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"HERRAMIENTA DE BASE DE DATOS LLAMADA: Obteniendo precio para {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"El precio del boleto a {city} es ${result[0]}" if result else "No hay datos de precio disponibles para esta ciudad"

In [ ]:
get_ticket_price("London")

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Ejercicio

¡Agrega una herramienta para establecer el precio de un boleto!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Aplicaciones de negocio</h2>
            <span style="color:#181;">¡Ojalá esto apenas necesite mencionarse! Ahora tienes la capacidad de dar acciones a tus LLMs. Este asistente de aerolínea ya puede hacer más que responder preguntas: ¡podría interactuar con APIs de reservas para hacer reservas!</span>
        </td>
    </tr>
</table>